In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini")
model.invoke([HumanMessage("잘 지냈어?")])

/Users/gwan/Desktop/code/AI-Agent/LLM_AI_Agent/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


AIMessage(content='네, 잘 지냈습니다! 당신은 어떻게 지내고 계신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 12, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ef804659a2', 'id': 'chatcmpl-DJehiTZUvfjHy8dSsaRakcbC76FSH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf16e-651c-7d31-83f3-60d8fc1c7923-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 18, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool 
def get_current_time(timezone:str, location: str) -> str:
    """현재 시각을 반환하는 함수

    Args: 
        timezone (str): 타임존(예: 'Asia/Seoul'). 실제 존재해야 함
        location (str): 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 model 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)

    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재 시각{now}'
    print(location_and_local_time)

    return location_and_local_time

In [6]:
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}


llm_with_tools = model.bind_tools(tools)

In [7]:
from langchain_core.messages import SystemMessage

messagse = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?"),
]

response = llm_with_tools.invoke(messagse)
messagse.append(response)

print(messagse)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 128, 'total_tokens': 151, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_58765056f8', 'id': 'chatcmpl-DJer2K9mIALEoNy0r99wHKM1akTeO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf177-363a-7e81-96f7-733ee6c253f7-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_vpynIRQV74aCWG0YPb3EgEKf', 'type': 'tool_call'}],

In [8]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict 사용해 도구 함수 선택
    print(tool_call["args"])                     # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call)   # 도구 함수를 호출해 결과 반환 
    messagse.append(tool_msg)

messagse

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각2026-03-15 21:30:06


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 128, 'total_tokens': 151, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_58765056f8', 'id': 'chatcmpl-DJer2K9mIALEoNy0r99wHKM1akTeO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf177-363a-7e81-96f7-733ee6c253f7-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_vpynIRQV74aCWG0YPb3EgEKf', 'type': 'tool_call'}

In [12]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="코식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간(예: 1d, 1mo, 1y)")
    

In [13]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()
    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = model.bind_tools(tools)

In [14]:
messagse.append(HumanMessage("테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messagse)
print(response)
messagse.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 274, 'total_tokens': 301, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1681c17ec', 'id': 'chatcmpl-DJf24WeYi6NFCgi8crKCOToYRHLQb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019cf181-a75d-7bd1-8af2-ed789355c807-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_WKLxtjtCCjZmOMotY5i0Uwg4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 274, 'output_tokens': 27, 'total_tokens': 301, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

In [17]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messagse.append(tool_msg)
    print(tool_msg)
    

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |    Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|--------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-02-17 00:00:00-05:00 | 412.36  | 413.72 | 400.51 |  410.63 | 5.96788e+07 |           0 |              0 |\n| 2026-02-18 00:00:00-05:00 | 411.11  | 416.9  | 409.58 |  411.32 | 4.59214e+07 |           0 |              0 |\n| 2026-02-19 00:00:00-05:00 | 407.25  | 415.25 | 404.11 |  411.71 | 5.10196e+07 |           0 |              0 |\n| 2026-02-20 00:00:00-05:00 | 408.3   | 414.7  | 405.5  |  411.82 | 5.79122e+07 |           0 |              0 |\n| 2026-02-23 00:00:00-05:00 | 407.29  | 407.7  | 394.04 |  399.83 | 6.968e+07   |           0 |              0 |\n| 2026-02-24 00:00:00-05:00 | 399.5   | 410.82 | 397.64 |  409.38 | 5.85795e+07 |           0 |              0 |\n| 2026-02-25 00:00

In [18]:
llm_with_tools.invoke(messagse)

AIMessage(content='테슬라(TSLA)의 주가는 한 달 전인 2026년 2월 17일에 비해 현재(2026년 3월 13일) 아래와 같이 변동했습니다.\n\n- **2026년 2월 17일 (1달 전)**: 종가 410.63\n- **2026년 3월 13일 (현재)**: 종가 391.20\n\n결론적으로, 테슬라의 주가는 한 달 전에 비해 하락했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 1476, 'total_tokens': 1587, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1681c17ec', 'id': 'chatcmpl-DJf5xYPTgbPEKfWkHpmTdxaPuJCl0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf185-5340-7c21-a1ba-b72b957c66eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1476, 'output_tokens': 111, 'total_tokens': 1587, 'input_token_details': {'audio': 0, 'cache_read